# 7.6. Convolutional Neural Networks \(LeNet\)

Let's look at the first CNN successfully trained via backpropagation on the original MNIST dataset known as LeNet. Back in the late 1980s to early 2000, such a convnet required writing and debugging months of C and assembly code, improving an existing Lisp-based deep learning library and performing trial and error over varying neural network architectures before finalizing the LeNet architecture after months of dedication and hard work.

In this chapter, we'll re-implement this classic using high-level APIs provided by MindSpore to appreciate how simple it is nowadays with modern, powerful deep learning libraries. We'll train our convnet on the Fashion MNIST dataset to confirm that it can reach about $90\%$ accuracy, a noticeable improvement over using simple linear models \($80\text{-}85\%$\) and plain MLP \($85\text{-}90\%$\).

In [1]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 7.6.1. LeNet

The original LeNet consisted of the following layers.

1. 1st convolution layer:
    1. 1 input channel
    1. 6 output channels
    1. $5 \times 5$ kernel with stride of 1
    1. Padding of `2px` in all directions
    1. Sigmoid activation
1. 1st average pooling layer with $2 \times 2$ window and a stride of 2
1. 2nd convolution layer:
    1. 6 input channel
    1. 16 output channels
    1. $5 \times 5$ kernel with stride of 1
    1. No padding
    1. Sigmoid activation
1. 2nd average pooling layer with $2 \times 2$ window and a stride of 2
1. Flattening layer to convert the $16 \times 5 \times 5$ feature map to $400$ input channels
1. 1st fully connected layer: 400 input channels, 120 output channels, sigmoid activation
1. 2nd fully connected layer: 120 input channels, 84 output channels, sigmoid activation
1. Final fully connected layer: 84 input channels, 10 output channels, Gaussian activation

The weights of each convolutional and fully connected layer are initialized based on Xavier initialization to avoid vanishing and exploding gradients with sigmoid activation.

We'll follow this design closely except we'll output raw logits for the final fully connected layer and apply softmax cross entropy with logits as our combined activation + loss function.

In [2]:
import mindspore.nn as nn

lenet = nn.SequentialCell([
    nn.Conv2d(1, 6, kernel_size=5, stride=1, weight_init='XavierUniform'),
    nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5, stride=1, pad_mode='valid', weight_init='XavierUniform'),
    nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Dense(400, 120, weight_init='XavierUniform', activation='sigmoid'),
    nn.Dense(120, 84, weight_init='XavierUniform', activation='sigmoid'),
    nn.Dense(84, 10)
])
lenet

SequentialCell(
  (0): Conv2d(input_channels=1, output_channels=6, kernel_size=(5, 5), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=XavierUniform, bias_init=None, format=NCHW)
  (1): Sigmoid()
  (2): AvgPool2d(kernel_size=2, stride=2, pad_mode=VALID)
  (3): Conv2d(input_channels=6, output_channels=16, kernel_size=(5, 5), stride=(1, 1), pad_mode=valid, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=XavierUniform, bias_init=None, format=NCHW)
  (4): Sigmoid()
  (5): AvgPool2d(kernel_size=2, stride=2, pad_mode=VALID)
  (6): Flatten()
  (7): Dense(
    input_channels=400, output_channels=120, has_bias=True, activation=Sigmoid()
    (activation): Sigmoid()
  )
  (8): Dense(
    input_channels=120, output_channels=84, has_bias=True, activation=Sigmoid()
    (activation): Sigmoid()
  )
  (9): Dense(input_channels=84, output_channels=10, has_bias=True)
)

To handle floating point precision issues automatically, let's wrap LeNet with MindSpore's AMP.

In [3]:
import mindspore.amp as amp

lenet_amp = amp.auto_mixed_precision(network=lenet, amp_level='O2')
lenet_amp

_OutputTo32(
  (_backbone): SequentialCell(
    (0): Conv2d(input_channels=1, output_channels=6, kernel_size=(5, 5), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=XavierUniform, bias_init=None, format=NCHW)
    (1): Sigmoid()
    (2): AvgPool2d(kernel_size=2, stride=2, pad_mode=VALID)
    (3): Conv2d(input_channels=6, output_channels=16, kernel_size=(5, 5), stride=(1, 1), pad_mode=valid, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=XavierUniform, bias_init=None, format=NCHW)
    (4): Sigmoid()
    (5): AvgPool2d(kernel_size=2, stride=2, pad_mode=VALID)
    (6): Flatten()
    (7): Dense(
      input_channels=400, output_channels=120, has_bias=True, activation=Sigmoid()
      (activation): Sigmoid()
    )
    (8): Dense(
      input_channels=120, output_channels=84, has_bias=True, activation=Sigmoid()
      (activation): Sigmoid()
    )
    (9): Dense(input_channels=84, output_channels=10, has_bias=True)
  )
)

Let's pass a grayscale image of size $28 \times 28$ pixels and inspect its shape after passing through each layer.

In [4]:
import mindspore.ops as ops

def layer_summary(net, X_shape):
    print(f'Input shape: {X_shape}')
    X = ops.randn(*X_shape)
    for cell in net.cells():
        X = cell(X)
        print(f'Output shape from {cell.__class__.__name__}: {X.shape}')

X_shape = (1, 1, 28, 28)
layer_summary(net=lenet, X_shape=X_shape)

Input shape: (1, 1, 28, 28)


/usr/local/Ascend/cann-8.5.0/python/site-packages/asc_op_compile_base/asc_op_compiler/ascendc_compile_gen_code.py:161: SyntaxWarning: invalid escape sequence '\w'
  match = re.search(f'{option}=(\w+)', ' '.join(compile_options))
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/c

Output shape from Conv2d: (1, 6, 28, 28)
Output shape from Sigmoid: (1, 6, 28, 28)
Output shape from AvgPool2d: (1, 6, 14, 14)
Output shape from Conv2d: (1, 16, 10, 10)
Output shape from Sigmoid: (1, 16, 10, 10)
Output shape from AvgPool2d: (1, 16, 5, 5)
Output shape from Flatten: (1, 400)
Output shape from Dense: (1, 120)
Output shape from Dense: (1, 84)
Output shape from Dense: (1, 10)


## 7.6.2. Training

Now to the interesting part - training LeNet on the Fashion MNIST dataset!

Let's download the dataset and load it with the usual transformations.

For the input images:

1. Resize all images to $28 \times 28$ pixels
1. Rescale all images by a factor of $\frac{1}{255}$ so all pixel values fall within the range $[0, 1]$
1. Swap the dimensions of each image from `(height, width, channels)` to `(channels, height, width)` expected by our convolution layers

For the output labels, we'll apply one-hot encoding and cast the resulting values to FP32.

In [5]:
import os

dataset_dir = 'data/fashion/'
os.makedirs(dataset_dir, exist_ok=True)

In [6]:
import gzip
import urllib.request

X_train_url = 'http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/train-images-idx3-ubyte.gz'
y_train_url = 'http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/train-labels-idx1-ubyte.gz'
X_test_url = 'http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/t10k-images-idx3-ubyte.gz'
y_test_url = 'http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/t10k-labels-idx1-ubyte.gz'

X_train_path = os.path.join(dataset_dir, 'train-images-idx3-ubyte')
y_train_path = os.path.join(dataset_dir, 'train-labels-idx1-ubyte')
X_test_path = os.path.join(dataset_dir, 't10k-images-idx3-ubyte')
y_test_path = os.path.join(dataset_dir, 't10k-labels-idx1-ubyte')

with urllib.request.urlopen(X_train_url) as response:
    with open(X_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_train_url) as response:
    with open(y_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(X_test_url) as response:
    with open(X_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_test_url) as response:
    with open(y_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

In [7]:
import mindspore.dataset as ds

train_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='train', shuffle=True)
test_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='test', shuffle=True)

In [8]:
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
from mindspore import dtype as mstype

def transform_ds(dataset):
    image_transforms = [
        vision.Resize(size=(28, 28)),
        vision.Rescale(rescale=1/255, shift=0),
        vision.HWC2CHW()
    ]
    label_transforms = [
        transforms.OneHot(num_classes=10),
        transforms.TypeCast(data_type=mstype.float32)
    ]
    dataset = dataset.map(operations=image_transforms, input_columns='image')
    dataset = dataset.map(operations=label_transforms, input_columns='label')
    dataset = dataset.batch(batch_size=128, drop_remainder=False)
    return dataset

train_ds = transform_ds(dataset=train_ds)
test_ds = transform_ds(dataset=test_ds)

We'll use the following hyperparameters for training.

1. Combined activation and loss function: softmax cross entropy with logits
1. Optimization algorithm: minibatch SGD with batch size of 128
1. Learning rate: `0.1`
1. Epochs: at most 100 with early stopping - stop early when validation loss no longer improves after 5 epochs and restore model weights from the best epoch

In [9]:
loss_fn = nn.SoftmaxCrossEntropyWithLogits(reduction='mean')
loss_fn

SoftmaxCrossEntropyWithLogits()

In [10]:
optimizer = nn.SGD(params=lenet_amp.trainable_params(), learning_rate=0.1)
optimizer

SGD()

In [11]:
from mindspore.train import Model

model = Model(network=lenet_amp, loss_fn=loss_fn, optimizer=optimizer, metrics={'accuracy', 'loss'})
model

In [12]:
from mindspore.train import EarlyStopping

early_stopping = EarlyStopping(patience=5, verbose=True, restore_best_weights=True)
early_stopping

In [13]:
epochs = 100

In [14]:
model.fit(epoch=epochs, train_dataset=train_ds, valid_dataset=test_ds, callbacks=[early_stopping])

...Restoring model weights from the end of the best epoch.
Epoch 00025: early stopping


As seen from above, LeNet reaches an optimal value for validation loss after around 15 epochs and stops early at around 20 epochs.

Let's check the validation loss and accuracy of our trained model based on LeNet.

In [15]:
val_metrics = model.eval(valid_dataset=test_ds)
val_loss = val_metrics['loss']
val_accuracy = val_metrics['accuracy']
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation accuracy: {val_accuracy:.4f}')

Validation loss: 0.6108
Validation accuracy: 0.7670


The accuracy is only about $75\%$, worse than our simple linear model. The results do match up with those obtained from [chapter 7.6](https://d2l.ai/chapter_convolutional-neural-networks/lenet.html) of the D2L textbook though.

Additionally, note that LeNet was designed for the original MNIST handwritten digits dataset where it obtained $> 99\%$ accuracy, a historic feat of its time.

## 7.6.3. Summary

We saw our first complete, working example of a convolutional neural network \(CNN\) in action known as LeNet. Back then, modern activation functions and pooling such as ReLU and max pooling were not discovered yet, hence the use of sigmoid activation and average pooling considered sub-optimal by today's standards. Nevertheless, LeNet marked a significant breakthrough in the use of CNNs for image classification and object recognition, by demonstrating the feasibility of such architectures in an era of rapidly evolving computer hardware.

In the next chapter, we'll see how modern convnets are defined and how they leverage the computing power of modern processors to provide much better accuracy on image-related tasks compared to traditional linear or MLP approaches.